# Shared Governance at Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/shared_governance_scale/shared_governance_scale.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/shared_governance_scale/shared_governance_scale.ipynb)

## Business Scenario

You are onboarding 500+ tables into Bronze and Silver. You need shared cleansing and validation rules without duplicating YAML across hundreds of contracts.

## Value Proposition

- Policy packs apply shared transformations and row rules
- Base templates standardize metadata, lineage, and dataset rules
- Soft-delete normalization across many tables
- Structured pivot/unpivot transformations keep analytics output consistent
- Registry-driven driver runs hundreds of contracts in parallel

---

## Goals

1. Apply a base template across many contracts
2. Use a shared policy pack for transformations
3. Run parallel ingestion with the driver


## Step 1: Review the Files

Key inputs in this example:

- contracts/_registry.yaml
- contracts/_shared/base_silver.yaml
- contracts/bronze/*.yaml (13 entities)
- contracts/silver/*.yaml (13 entities)
- policy_packs/shared_standard.yaml
- data/*.csv (13 entities)


## Step 2: Apply the Base Template

This deep-merges shared defaults into all Silver contracts, appends shared list rules, and injects
soft-delete handling when operation/deleted_at/is_deleted columns exist.


In [ ]:
!python scripts/apply_contract_template.py \
  --base-template examples/06_advanced_workflows/shared_governance_scale/contracts/_shared/base_silver.yaml \
  --registry examples/06_advanced_workflows/shared_governance_scale/contracts/_registry.yaml \
  --stage silver \
  --list-merge-keys transformations,quality.row_rules,quality.dataset_rules \
  --list-mode append \
  --soft-delete

## Step 2b: Apply Template via Python (Optional)

Use the same shared template logic without the CLI.


In [ ]:
from pathlib import Path

from lakelogic.tools.template_apply import apply_contract_template

apply_contract_template(
    base_template=Path("examples/06_advanced_workflows/shared_governance_scale/contracts/_shared/base_silver.yaml"),
    registry=Path("examples/06_advanced_workflows/shared_governance_scale/contracts/_registry.yaml"),
    stage="silver",
    list_merge_keys=["transformations", "quality.row_rules", "quality.dataset_rules"],
    list_mode="append",
    soft_delete=True,
)


## Step 3: Run in Parallel with a Shared Policy Pack

Policy packs add shared transformations and row rules for every contract (13 entities in this demo).


In [ ]:
!lakelogic-driver \
  --registry examples/06_advanced_workflows/shared_governance_scale/contracts/_registry.yaml \
  --layers bronze,silver \
  --policy-pack shared_standard \
  --policy-pack-dir examples/06_advanced_workflows/shared_governance_scale/policy_packs \
  --max-workers 8

## Step 3b: Run the Driver in Python (Optional)

This mirrors the CLI invocation using the `PipelineDriver` API.


In [ ]:
from pathlib import Path

from lakelogic.cli.driver import PipelineDriver, Window

registry_paths = {
    "system": Path("examples/06_advanced_workflows/shared_governance_scale/contracts/_registry.yaml"),
    "reference": None,
    "gold": None,
}

driver = PipelineDriver(
    engine="polars",
    max_workers=8,
    policy_pack="shared_standard",
    policy_pack_dir=Path("examples/06_advanced_workflows/shared_governance_scale/policy_packs"),
)

driver.run(
    registry_paths,
    layers=["bronze", "silver"],
    window=Window(None, None, "full"),
    reprocess=False,
)


## Step 4: Structured Pivot + Unpivot

- `silver_feature_flags.yaml` uses `pivot` to create per-user feature flag columns.
- `silver_daily_metrics.yaml` uses `unpivot` to normalize wide metrics into long format.


## Step 5: Inspect Outputs

After running, load the output Parquet files using a LakeLogic engine (Polars by default).


In [ ]:
from pathlib import Path

from lakelogic.core.processor import DataProcessor

BASE = Path("examples/06_advanced_workflows/shared_governance_scale/output/silver")

def load_silver(name: str):
    contract = {
        "info": {"title": f"Inspect {name}"},
        "dataset": f"inspect_{name}",
        "source": {"type": "landing", "path": str(BASE / name / "*.parquet"), "load_mode": "full"},
        "quality": {"enforce_required": False},
    }
    processor = DataProcessor(contract, engine="polars")
    result = processor.run_source()
    return result.good

users = load_silver("users")
events = load_silver("events")
flags = load_silver("feature_flags")
metrics = load_silver("daily_metrics")

print(users.head())
print(events.head())
print(flags.head())
print(metrics.head())
